# MapClass — Dataset Inspection

Visual + statistical inspection of the GCS-canonical datasets built by
`build_historical_dataset.py` and `build_satellite_dataset.py`.

Each sample lives at `gs://mapclass-training-northeast1/data/<family>/dataset/<id>/`
with a `pyramids/py_rNNN_cNNN/` subdir per pyramid. Every pyramid holds a
21-tile nest: one `896`, four `448_<ci>`, sixteen `224_<ci>_<gi>`, each as an
`{image,land_cover,topography}.png` triplet, plus `pyramid.json` and
`sample_weights.json`.

`land_cover.png` / `topography.png` are single-channel uint8 **class-index**
images (not RGB) — `255` is NODATA. The canonical taxonomy is imported live
from `scripts/biome_mapping.py` so this notebook never drifts from the code.

In [ ]:
import io, json, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from PIL import Image
import gcsfs

# Import the canonical taxonomy + GCS constants straight from the repo so
# this notebook stays in lockstep with the build pipeline.
_REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(_REPO / 'scripts'))
from biome_mapping import LANDCOVER_CLASSES, TOPO_CLASSES  # noqa: E402
import gcs_io  # noqa: E402

fs = gcsfs.GCSFileSystem(project=gcs_io.GCS_PROJECT)
DATA = gcs_io.DATA_PREFIX  # 'mapclass-training-northeast1/data'

print('land cover:', LANDCOVER_CLASSES)
print('topography:', TOPO_CLASSES)
print('GCS root :', DATA)

In [ ]:
# Palettes (index -> RGB). 255 (NODATA) renders black.
LC_COLORS = {
    'water': '#3a7ca5', 'trees': '#1b7a3d', 'shrubland': '#9bbf4f',
    'grassland': '#b7d96b', 'cropland': '#e8c060', 'built_up': '#d1495b',
    'bare_sparse': '#c2a878', 'flooded_wetland': '#5fa8a0', 'snow_ice': '#eef0f5',
}
TOPO_COLORS = {'flat': '#cfe8cf', 'hilly': '#d8b878', 'mountainous': '#8a5a3c'}


def _lut(class_names, color_map):
    """256x3 uint8 lookup table; unknown/NODATA -> black."""
    lut = np.zeros((256, 3), np.uint8)
    for i, name in enumerate(class_names):
        c = color_map[name].lstrip('#')
        lut[i] = [int(c[0:2], 16), int(c[2:4], 16), int(c[4:6], 16)]
    return lut


LC_LUT = _lut(LANDCOVER_CLASSES, LC_COLORS)
TOPO_LUT = _lut(TOPO_CLASSES, TOPO_COLORS)


def colorize(idx_arr, lut):
    return lut[idx_arr]


def legend_handles(class_names, color_map, present=None):
    items = class_names if present is None else [class_names[i] for i in present]
    return [Patch(facecolor=color_map[n], edgecolor='gray', label=n) for n in items]

In [ ]:
# --- GCS helpers -----------------------------------------------------------
def list_samples(family):
    """family in {'historical','satellite'} -> sorted list of sample ids."""
    root = f'{DATA}/{family}/dataset'
    return sorted(
        p.rstrip('/').split('/')[-1]
        for p in fs.ls(root)
        if fs.isdir(p)
    )


def list_pyramids(family, sample):
    base = f'{DATA}/{family}/dataset/{sample}/pyramids'
    return sorted(p.rstrip('/').split('/')[-1] for p in fs.ls(base) if fs.isdir(p))


def load_png(family, sample, pyramid, tile_id, kind):
    """kind in {'image','land_cover','topography'} -> numpy array."""
    path = f'{DATA}/{family}/dataset/{sample}/pyramids/{pyramid}/{tile_id}_{kind}.png'
    with fs.open(path, 'rb') as fh:
        return np.array(Image.open(io.BytesIO(fh.read())))


def load_json(family, sample, pyramid, name):
    path = f'{DATA}/{family}/dataset/{sample}/pyramids/{pyramid}/{name}'
    with fs.open(path, 'r') as fh:
        return json.load(fh)


hist = list_samples('historical')
sat = list_samples('satellite')
print(f'historical samples: {len(hist)}')
print('  ' + ', '.join(hist))
print(f'satellite samples : {len(sat)}')
print('  ' + ', '.join(sat[:8]) + (' ...' if len(sat) > 8 else ''))

## 1. Inspect a single sample

Shows the `896` context tile of one pyramid as a 3-panel: RGB image,
colorized land cover, colorized topography. Change `FAMILY` / `SAMPLE` /
`PYRAMID_IDX` / `TILE_ID` to browse (e.g. `TILE_ID='224_0_0'` for a 224 crop).

In [ ]:
FAMILY = 'satellite'      # 'historical' or 'satellite'
SAMPLE = None             # None -> first sample of FAMILY;
                          # OR a full id string from `hist`/`sat`,
                          # e.g. 'RUMSEY_8_1_369928_90137299__plate0'
PYRAMID_IDX = 0           # which pyramid in the sample
TILE_ID = '896'           # '896' | '448_<0-3>' | '224_<0-3>_<0-3>'

samples = hist if FAMILY == 'historical' else sat
sample = SAMPLE or samples[0]
pyramids = list_pyramids(FAMILY, sample)
pyr = pyramids[PYRAMID_IDX]

img = load_png(FAMILY, sample, pyr, TILE_ID, 'image')
lc = load_png(FAMILY, sample, pyr, TILE_ID, 'land_cover')
topo = load_png(FAMILY, sample, pyr, TILE_ID, 'topography')

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(img); ax[0].set_title(f'{FAMILY} / {sample}\n{pyr} / {TILE_ID} — image')
ax[1].imshow(colorize(lc, LC_LUT)); ax[1].set_title('land cover')
ax[2].imshow(colorize(topo, TOPO_LUT)); ax[2].set_title('topography')
for a in ax:
    a.axis('off')

lc_present = sorted(int(v) for v in np.unique(lc) if v < len(LANDCOVER_CLASSES))
topo_present = sorted(int(v) for v in np.unique(topo) if v < len(TOPO_CLASSES))
ax[1].legend(handles=legend_handles(LANDCOVER_CLASSES, LC_COLORS, lc_present),
             loc='lower center', bbox_to_anchor=(0.5, -0.32), ncol=3, fontsize=8)
ax[2].legend(handles=legend_handles(TOPO_CLASSES, TOPO_COLORS, topo_present),
             loc='lower center', bbox_to_anchor=(0.5, -0.22), ncol=3, fontsize=8)
plt.tight_layout(); plt.show()

nodata = float((lc == 255).mean()) * 100
print(f'land-cover classes present: {[LANDCOVER_CLASSES[i] for i in lc_present]}')
print(f'topography classes present: {[TOPO_CLASSES[i] for i in topo_present]}')
print(f'NODATA (255) in land cover: {nodata:.1f}% of pixels')

## 2. Class distribution for a sample

Aggregates pixel counts across the `896` tile of **every** pyramid in the
sample (non-overlapping at 896 stride-448 → a fair sample-level histogram).

In [ ]:
def class_histogram(family, sample, n_classes, kind):
    counts = np.zeros(n_classes + 1, np.int64)  # last bin = NODATA/other
    for pyr in list_pyramids(family, sample):
        arr = load_png(family, sample, pyr, '896', kind).ravel()
        valid = arr[arr < n_classes]
        counts[:n_classes] += np.bincount(valid, minlength=n_classes)
        counts[n_classes] += int((arr >= n_classes).sum())
    return counts


lc_counts = class_histogram(FAMILY, sample, len(LANDCOVER_CLASSES), 'land_cover')
topo_counts = class_histogram(FAMILY, sample, len(TOPO_CLASSES), 'topography')

fig, ax = plt.subplots(1, 2, figsize=(16, 5))
for a, names, colors, counts, title in (
    (ax[0], LANDCOVER_CLASSES, LC_COLORS, lc_counts, 'land cover'),
    (ax[1], TOPO_CLASSES, TOPO_COLORS, topo_counts, 'topography'),
):
    labels = names + ['nodata']
    bar_colors = [colors[n] for n in names] + ['#222222']
    frac = counts / counts.sum() * 100
    a.bar(labels, frac, color=bar_colors, edgecolor='gray')
    a.set_title(f'{title} — {FAMILY}/{sample} ({len(list_pyramids(FAMILY, sample))} pyramids)')
    a.set_ylabel('% of pixels')
    a.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

## 3. Scan one map — N tiles from a single sample

Shows N pyramid `896` tiles from **one** map so you can sweep across it and
spot non-terrain regions (title pages, cartouches, bindings, blank margins).
Pyramids are listed row-major (`py_rROW_cCOL`), so list order ≈ a
top-to-bottom spatial scan of the source.

**How to pick a map:**
- The available ids were printed by the helpers cell above:
  `hist` (4 historical maps) and `sat` (199 satellite samples).
- Set `SCAN_SAMPLE` to either an **int index** into that list
  (`0` = first) **or** the **full id string**
  (e.g. `'RUMSEY_8_1_369928_90137299__plate0'`).
- Each tile's title is `#<pyramid index>  <pyramid id>` — copy the index
  into Section 1's `PYRAMID_IDX` to drill into that tile's full triplet.
- Big maps: page through with `PYRAMID_START` (e.g. `8`, `16`, …).

In [ ]:
# --- pick what to scan -----------------------------------------------------
SCAN_FAMILY   = 'historical'   # 'historical' or 'satellite'
SCAN_SAMPLE   = 3              # int index into hist/sat  OR  full id string
N_TILES       = 8              # how many pyramid 896-tiles to show (default 8)
SCAN_KIND     = 'image'        # 'image' | 'land_cover' | 'topography'
PYRAMID_START = 12              # offset into the pyramid list (page large maps)

_samples = hist if SCAN_FAMILY == 'historical' else sat
scan_sample = (_samples[SCAN_SAMPLE] if isinstance(SCAN_SAMPLE, int)
               else SCAN_SAMPLE)
all_pyr = list_pyramids(SCAN_FAMILY, scan_sample)
sel = all_pyr[PYRAMID_START:PYRAMID_START + N_TILES]
print(f'{SCAN_FAMILY} / {scan_sample}: {len(all_pyr)} pyramids total; '
      f'showing {len(sel)} from #{PYRAMID_START}')

cols = 4
rows = int(np.ceil(len(sel) / cols)) or 1
fig, ax = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
ax = np.atleast_2d(ax)
for k, pyr in enumerate(sel):
    r, c = divmod(k, cols)
    idx = PYRAMID_START + k          # index to reuse as Section 1 PYRAMID_IDX
    try:
        arr = load_png(SCAN_FAMILY, scan_sample, pyr, '896', SCAN_KIND)
        if SCAN_KIND == 'land_cover':
            arr = colorize(arr, LC_LUT)
        elif SCAN_KIND == 'topography':
            arr = colorize(arr, TOPO_LUT)
        ax[r, c].imshow(arr)
        ax[r, c].set_title(f'#{idx}  {pyr}', fontsize=8)
    except Exception as e:  # noqa: BLE001 — one bad tile must not abort the scan
        ax[r, c].set_title(f'#{idx} {pyr}\n{type(e).__name__}',
                           fontsize=7, color='red')
    ax[r, c].axis('off')
for k in range(len(sel), rows * cols):
    r, c = divmod(k, cols)
    ax[r, c].axis('off')
fig.suptitle(f'{SCAN_FAMILY} / {scan_sample} — {SCAN_KIND} '
             f'(pyramids {PYRAMID_START}–{PYRAMID_START + len(sel) - 1} '
             f'of {len(all_pyr)})', y=1.02)
plt.tight_layout(); plt.show()

## 4. Per-sample metadata: `sample_weights.json` + `pyramid.json`

`sample_weights.json` carries the class-conditional per-source loss weights
(historical downweights land-use classes; satellite upweights the
satellite-only classes). `pyramid.json` is the 21-tile manifest with
parent→child indices.

In [ ]:
pyr0 = list_pyramids(FAMILY, sample)[0]
weights = load_json(FAMILY, sample, pyr0, 'sample_weights.json')
manifest = load_json(FAMILY, sample, pyr0, 'pyramid.json')

print(f'sample_weights.json ({FAMILY}/{sample}):')
print(json.dumps(weights, indent=2))

tiles = manifest.get('tiles', manifest if isinstance(manifest, list) else [])
print(f'\npyramid.json: {len(tiles)} tile entries (expect 21)')
for t in tiles[:3]:
    print('  ', {k: t[k] for k in ('id', 'x', 'y', 'size') if k in t})